## 키포인트 추출

In [4]:
import os
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tqdm import tqdm  

# 데이터 경로 설정
video_path = "/home/wonho/data/store/Training/01.원천데이터/TS_03.이상행동_08.파손"
label_path = "/home/wonho/data/store/Training/02.라벨링데이터/TL_03.이상행동_08.파손"

# Mediapipe 초기화
try:
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose()
except AttributeError:
    raise RuntimeError("Mediapipe와 numpy 간의 호환성 문제 발생. numpy 버전을 확인하세요.")

def extract_keypoints(video_file):
    cap = cv2.VideoCapture(video_file)
    keypoints = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Mediapipe를 사용하여 키포인트 추출
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        try:
            results = pose.process(frame_rgb)
        except AttributeError:
            raise RuntimeError("Mediapipe 처리 중 문제가 발생했습니다. numpy와 mediapipe 버전을 확인하세요.")

        if results.pose_landmarks:
            keypoints.append(
                np.array([[lm.x, lm.y, lm.z] for lm in results.pose_landmarks.landmark]).flatten()
            )
        else:
            keypoints.append(np.zeros(33 * 3))  # 키포인트가 없으면 0으로 채움

    cap.release()
    return np.array(keypoints)

# 예제: 첫 번째 영상 처리
try:
    video_files = [os.path.join(video_path, f) for f in os.listdir(video_path) if f.endswith(".mp4")]
    video_files = video_files[:20]  # 첫 20개만 선택
    keypoints_data = [extract_keypoints(video) for video in tqdm(video_files, desc="Processing Videos")]
except FileNotFoundError:
    raise RuntimeError(f"데이터 경로가 올바르지 않습니다: {video_path}")

I0000 00:00:1749103849.664950   20453 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1749103849.667109   23093 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
Processing Videos:   0%|          | 0/20 [00:00<?, ?it/s]

W0000 00:00:1749103849.731886   23090 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1749103849.763253   23082 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Processing Videos: 100%|██████████| 20/20 [01:46<00:00,  5.34s/it]


## LSTM 모델 학습

[{'label': 'Right foot', 'frame': '68', 'coordinates': '1120,576'},
 {'label': 'Right foot', 'frame': '69', 'coordinates': '1120,576'},
 {'label': 'Right knee', 'frame': '68', 'coordinates': '1112,501'},
 {'label': 'Right knee', 'frame': '69', 'coordinates': '1112,501'},
 {'label': 'Right  hip', 'frame': '68', 'coordinates': '1097,422'},
 {'label': 'Right  hip', 'frame': '69', 'coordinates': '1097,422'},
 {'label': 'Left hip', 'frame': '68', 'coordinates': '1051,414'},
 {'label': 'Left hip', 'frame': '69', 'coordinates': '1051,414'},
 {'label': 'Left knee', 'frame': '68', 'coordinates': '1043,502'},
 {'label': 'Left knee', 'frame': '69', 'coordinates': '1043,502'},
 {'label': 'Left foot', 'frame': '68', 'coordinates': '1036,583'},
 {'label': 'Left foot', 'frame': '69', 'coordinates': '1036,583'},
 {'label': 'Pelvis', 'frame': '68', 'coordinates': '1073,418'},
 {'label': 'Pelvis', 'frame': '69', 'coordinates': '1073,418'},
 {'label': 'Neck base', 'frame': '68', 'coordinates': '1106,271'

In [ ]:
for i in range(len(keypoints_data)): 
    keypoints_data[i] = np.resize(keypoints_data[i], (180, 99))
    print(keypoints_data[i].shape)

(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)
(180, 99)


In [31]:
import os

# 디렉토리 내 파일 확인
label_files = [f for f in os.listdir(label_path) if f.endswith(".xml")]
print(label_files[:20])

['C_3_8_7_BU_SMA_09-07_14-35-52_CA_RGB_DF2_M2.xml', 'C_3_8_42_BU_SMC_10-14_12-02-30_CD_RGB_DF2_F2.xml', 'C_3_8_2_BU_SMC_08-07_12-22-04_CA_RGB_DF2_M1.xml', 'C_3_8_30_BU_SYA_10-06_12-41-40_CB_RGB_DF2_F3.xml', 'C_3_8_36_BU_SMC_10-14_10-03-28_CB_RGB_DF2_M2.xml', 'C_3_8_25_BU_SYB_10-04_11-41-08_CD_RGB_DF2_F3.xml', 'C_3_8_8_BU_DYB_08-10_13-21-42_CD_RGB_DF2_M2.xml', 'C_3_8_12_BU_SMB_09-01_13-02-32_CA_RGB_DF2_M2.xml', 'C_3_8_29_BU_SMB_09-02_13-53-36_CB_RGB_DF2_F3.xml', 'C_3_8_52_BU_DYB_10-17_11-12-07_CC_RGB_DF2_F2.xml', 'C_3_8_2_BU_SMB_09-17_10-54-25_CB_RGB_DF2_M1.xml', 'C_3_8_43_BU_SMC_10-14_12-04-41_CE_RGB_DF2_F2.xml', 'C_3_8_21_BU_SMB_09-02_15-33-25_CD_RGB_DF2_M3.xml', 'C_3_8_23_BU_SYA_10-06_12-31-21_CD_RGB_DF2_M3.xml', 'C_3_8_11_BU_DYB_08-10_14-41-24_CF_RGB_DF2_F2.xml', 'C_3_8_23_BU_SYB_10-04_10-49-52_CD_RGB_DF2_M3.xml', 'C_3_8_18_BU_SMB_09-01_14-36-34_CD_RGB_DF2_F2.xml', 'C_3_8_32_BU_SMB_09-05_13-10-16_CD_RGB_DF2_M4.xml', 'C_3_8_8_BU_DYB_08-10_13-21-47_CF_RGB_DF2_M2.xml', 'C_3_8_47_BU_DYB

In [34]:
import xml.etree.ElementTree as ET

def extract_keypoints_from_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    keypoints = []
    for track in root.findall(".//track"):
        label = track.get("label")
        for points in track.findall(".//points"):
            frame = points.get("frame")
            coordinates = points.get("points")
            keypoints.append({"label": label, "frame": frame, "coordinates": coordinates})
    
    return keypoints



In [36]:
# Example usage
xml_file_path = "/home/wonho/data/store/Training/02.라벨링데이터/TL_03.이상행동_08.파손/C_3_8_1_BU_SMA_09-17_13-38-51_CA_RGB_DF2_M1.xml"
keypoints_data = extract_keypoints_from_xml(xml_file_path)
print(keypoints_data)  # Print first 5 keypoints
print(len(keypoints_data))

[{'label': 'Right foot', 'frame': '68', 'coordinates': '1120,576'}, {'label': 'Right foot', 'frame': '69', 'coordinates': '1120,576'}, {'label': 'Right knee', 'frame': '68', 'coordinates': '1112,501'}, {'label': 'Right knee', 'frame': '69', 'coordinates': '1112,501'}, {'label': 'Right  hip', 'frame': '68', 'coordinates': '1097,422'}, {'label': 'Right  hip', 'frame': '69', 'coordinates': '1097,422'}, {'label': 'Left hip', 'frame': '68', 'coordinates': '1051,414'}, {'label': 'Left hip', 'frame': '69', 'coordinates': '1051,414'}, {'label': 'Left knee', 'frame': '68', 'coordinates': '1043,502'}, {'label': 'Left knee', 'frame': '69', 'coordinates': '1043,502'}, {'label': 'Left foot', 'frame': '68', 'coordinates': '1036,583'}, {'label': 'Left foot', 'frame': '69', 'coordinates': '1036,583'}, {'label': 'Pelvis', 'frame': '68', 'coordinates': '1073,418'}, {'label': 'Pelvis', 'frame': '69', 'coordinates': '1073,418'}, {'label': 'Neck base', 'frame': '68', 'coordinates': '1106,271'}, {'label': '

In [ ]:
X = np.array(keypoints_data)  # 키포인트 데이터 (패딩 또는 잘라내기 완료된 상태)
y = np.array(label_data)      # 라벨 데이터 (XML에서 추출한 값)

# 데이터셋 분할
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

IsADirectoryError: [Errno 21] Is a directory: '/home/wonho/data/store/Training/02.라벨링데이터/TL_03.이상행동_08.파손'

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

def create_lstm_model(input_shape):
    model = Sequential([
        LSTM(128, return_sequences=True, activation='relu', input_shape=input_shape),
        LSTM(64, return_sequences=False, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')  # 이진 분류 (파손 여부)
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# 모델 생성
input_shape = (X_train.shape[1], X_train.shape[2])  # 시계열 데이터의 형태
model = create_lstm_model(input_shape)